# 20 - Simplified Modeling: High-Performance Vectorized Fleets & Ensembles

A central challenge in energy system co-simulation is complexity:
> **"Some of the modeling work, esp the vectorized models and ensemble models have complex class definitions. Can we make it simpler?"**

Yes! While `energysim` provides low-level base classes (`VectorizedContainerAdapter`, `Sampler`), writing 40 lines of boilerplate class definition just to simulate a fleet of electric vehicles or run a parameter sweep creates unnecessary friction.

In this tutorial, you will discover the **simplified, declarative modeling APIs**:
1. **`SimpleVectorizedModel`**: Define fleets of 1,000s of Distributed Energy Resources (DERs) in a few lines of functional NumPy code.
2. **`Ensemble.sweep`**: Execute full-factorial scenario studies and Monte Carlo ensembles without manual scenario loops or factory boilerplate.


In [ ]:
from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from energysim import world, SimpleVectorizedModel, Ensemble

print("Simplified modeling interfaces imported successfully!")


## 1. Simplified Vectorized Fleets: From 40 Lines to 6

In traditional co-simulation frameworks, simulating a fleet of 2,000 electric vehicles requires creating 2,000 Python objects, resulting in massive interpreter overhead.

`energysim` solves this with vectorized container simulators running at C-speed in pure NumPy. With `SimpleVectorizedModel`, you no longer need to write a custom class:


In [ ]:
FLEET_SIZE = 2000

# Heterogeneous fleet parameters (NumPy arrays)
rng = np.random.default_rng(42)
capacities_kwh = rng.uniform(40.0, 90.0, size=FLEET_SIZE)
initial_soc = rng.uniform(0.15, 0.50, size=FLEET_SIZE)

# 1. Define a pure vectorized NumPy step function (in-place update)
def step_ev_fleet(states, inputs, dt, time):
    soc = states["soc"]
    p_req = inputs["p_charge_kw"]  # scalar broadcast or vector setpoint
    
    # Physical constraints: stop charging when battery is full (SoC >= 0.95)
    can_charge = soc < 0.95
    actual_p = np.where(can_charge, np.clip(p_req, 0.0, 11.0), 0.0)
    
    # Integrate State of Charge: dSoC = P * eta * dt / Capacity
    eta = 0.94
    delta_soc = (actual_p * eta * (dt / 3600.0)) / capacities_kwh
    states["soc"][:] = np.clip(soc + delta_soc, 0.0, 1.0)
    states["actual_p_kw"][:] = actual_p

# 2. Instantiate with SimpleVectorizedModel
ev_fleet = SimpleVectorizedModel(
    name="ev_fleet",
    size=FLEET_SIZE,
    states={"soc": initial_soc, "actual_p_kw": 0.0},
    inputs={"p_charge_kw": 0.0},
    step_fn=step_ev_fleet,
    step_size=300.0,
    aliases={"power": "actual_p_kw"}
)

print(f"Created vectorized fleet of {FLEET_SIZE:,} EV entities!")
print("Automatic aggregate outputs available:")
print([v for v in ev_fleet.get_available_variables()["outputs"] if v.startswith("total_") or v.startswith("mean_")])


## 2. Co-Simulating the Vectorized Fleet

Notice that `SimpleVectorizedModel` automatically computes:
- `total_<var>` (e.g. `total_actual_p_kw` gives aggregate fleet grid demand)
- `mean_<var>`, `min_<var>`, `max_<var>` (e.g. `mean_soc`, `min_soc`)
- Individual indexing `soc[0]`, `soc[42]`

Let's simulate a 6-hour charging session on the `world` canvas:


In [ ]:
sim = world(start_time=0, stop_time=3600 * 6, t_macro=900, power_unit="kW")

sim.add_container("evs", ev_fleet)

# Dynamic charging signal (e.g. utility demand response signal)
sim.add_signal("charge_setpoint", lambda t: 11.0 if t < 3600 * 3 else 7.4, step_size=900, units={"y": "kW"})

sim.add_connections({
    "charge_setpoint.y": "evs.p_charge_kw"
})

res = sim.simulate(pbar=False)
df = res.to_frame()

print("Fleet simulation complete!")
display(df[["charge_setpoint.y", "evs.total_actual_p_kw", "evs.mean_soc"]].head())


## 3. Visualizing Fleet Aggregation

Let's plot the total power drawn by all 2,000 EVs and the resulting average State of Charge:


In [ ]:
hours = df.index / 3600.0

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

ax1.plot(hours, df["evs.total_actual_p_kw"] / 1000.0, color="navy", lw=2, label="Aggregate Grid Power [MW]")
ax1.set_ylabel("Power [MW]")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_title(f"High-Speed Vectorized Co-Simulation: Fleet of {FLEET_SIZE:,} EVs", fontsize=11)

ax2.plot(hours, df["evs.mean_soc"] * 100.0, color="green", lw=2, label="Average Fleet SoC [%]")
ax2.axhline(95.0, color="red", ls="--", lw=0.8, label="Full Target (95%)")
ax2.set_ylabel("State of Charge [%]")
ax2.set_xlabel("Time [hours]")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 4. Simplified Ensembles: 1-Line Full Factorial Sweep

Running Monte Carlo studies or sensitivity sweeps usually requires boilerplate loops, scenario builders, and result extractors.

With `Ensemble.sweep`, you define a clean factory function that takes plain Python keyword arguments, and pass a dictionary of parameter lists:


In [ ]:
# 1. Define a clean world builder accepting keyword arguments
def build_microgrid_study(p_charge=11.0, duration_h=4.0):
    w = world(start_time=0, stop_time=int(duration_h * 3600), t_macro=900)
    fleet = SimpleVectorizedModel(
        name="evs",
        size=FLEET_SIZE,
        states={"soc": 0.20, "actual_p_kw": 0.0},
        inputs={"p_charge_kw": float(p_charge)},
        step_fn=step_ev_fleet
    )
    w.add_container("evs", fleet)
    return w

# 2. Run a full-factorial parameter sweep in one line
study = Ensemble.sweep(
    build_microgrid_study,
    params={
        "p_charge": [3.7, 7.4, 11.0, 22.0],  # 4 charging power levels
        "duration_h": [2.0, 4.0, 6.0]         # 3 session durations
    },
    keep="kpis"
).run()

# 3. Inspect results
kpis = study.kpis()
print(f"Sweep complete! Ran {len(kpis)} scenarios.")
display(kpis[["evs.total_actual_p_kw:mean", "evs.mean_soc:max"]].head(8))


## 5. Summary of Simplifications

| Feature | Legacy / Verbose Pattern | New Simplified API |
| :--- | :--- | :--- |
| **Vectorized Fleets** | Subclass `VectorizedContainerAdapter`, write `__init__`, `register_state`, `register_input`, manual snapshots | `SimpleVectorizedModel(name, size, states, inputs, step_fn)` |
| **Fleet Update Logic** | Class method overriding `step(self, time, step_size)` | Plain functional callback `step_fn(states, inputs, dt, time)` |
| **Grid Parameter Sweeps** | Manual `Scenarios` cartesian product, `_MapFactory`, `lambda params, seed:` | `Ensemble.sweep(builder, params={'a': [...], 'b': [...]})` |
